# Weka gateway benchmarking

Generate output JSON that we can then aggregate, plot, and do statistics on.

## Methodology

This test uses the Butler to get handles for a large number of queries from one or more datasets.  Acquiring the query handles does not pull the actual query result data.  We then select a much smaller sample of those queries to retrieve the result data.

For each query within a given dataset, we measure the size of the HTTP object (referenced with a signed URL via the Weka-S3 gateway at USDF) to retrieve, and fetch the object.  The fetch is performed with `curl`, because we believe that `curl` redirected to `/dev/null` will be much faster than pulling an HTTP object with a Python library.

After all objects have been fetched, the retrieval is (optionally) repeated, to give us insight into the cache size and replacement policy.

This process is then repeated for each dataset.

Statistics (mean, median, population variance, population standard deviation, and coefficient of variation) are collected for each of: object size, initial pull speed, repeated pull speed, and ratio of re-checked speed to initial speed.

This is then persisted into a file in a shared directory, so we can eventually aggregate over those files, allowing us to test performance of many retrievals in parallel via the `mobu` testing framework.

In [1]:
# Imports
import json
from pathlib import Path

from weka_helpers import (
    Dataset,
    convert_results_to_primitive,
    get_results
)

## Set up input parameters

In [2]:
# Set up report output directory
output_dir=Path("/rubin/bench/weka")  # Only on data-int
# Set this to read results from a file rather than test speeds
input_filename: Path|None = None

# Butler setup
butler_config="dp2"
butler_collections="dp2"

# Unset this to *not* repeat the test with the same files
#  (to test whether Weka caching affects speed)
recheck_speeds = True

# Define our input data parameters

data_map: list[Dataset] = [ 
    Dataset(
        display_name = "datasets",
        dataset_name = "visit_image",  # 30 MB each-ish
        count = 5,
        limit = 10_000,
    ),
    Dataset(
        display_name = "big_datasets",
        dataset_name = "run_provenance",  # 1GB each-ish, variable
        count = 0,
        limit = 10_000,
    ),
]

# anything else with a "+" on this page is also hosted at USDF: https://rubinobs.atlassian.net/wiki/x/CgCveg
# (anything without a +/pencil is on GCS)

## Fetch results

In [3]:
res=get_results(
    butler_config=butler_config,
    butler_collections=butler_collections,
    data_map=data_map,
    output_dir=output_dir,
    input_filename=input_filename,
    recheck_speeds=recheck_speeds)

print(json.dumps(convert_results_to_primitive(res),
        sort_keys=True,
        indent=2))

Butler query of datasets(visit_image), limit=10000...sampling 5 records
Getting size for datasets: 1/5...22852800 bytes
Getting size for datasets: 2/5...26723520 bytes
Getting size for datasets: 3/5...24154560 bytes
Getting size for datasets: 4/5...24361920 bytes
Getting size for datasets: 5/5...20957760 bytes
Getting speed for datasets: 1/5...3971072.0 bytes/s
Getting speed for datasets: 2/5...4794368.0 bytes/s
Getting speed for datasets: 3/5...4986880.0 bytes/s
Getting speed for datasets: 4/5...4322304.0 bytes/s
Getting speed for datasets: 5/5...4753408.0 bytes/s
Getting speed for datasets: 1/5...18140364.8 bytes/s
Getting speed for datasets: 2/5...6481920.0 bytes/s
Getting speed for datasets: 3/5...18559795.2 bytes/s
Getting speed for datasets: 4/5...1957888.0 bytes/s
Getting speed for datasets: 5/5...4239360.0 bytes/s
[
  {
    "dataset_name": "visit_image",
    "display_name": "datasets",
    "initial_statistics": {
      "CV": 0.08063954676988257,
      "mean": 4565606.4,
      "